In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

In [2]:
!conda activate seg2d
!python -c "import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))"

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Imports and config

In [3]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score,
    f1_score, average_precision_score, roc_auc_score
)

# ── reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── hyperparameters ───────────────────────────────────────────────────────────
WINDOW_SIZE     = 30
RUL_CAP         = 125
FAULT_THRESHOLD = 30
FL_ROUNDS       = 25
LOCAL_EPOCHS    = 8
BATCH_SIZE      = 64
DROP_SENSORS    = ["s1", "s5", "s10", "s16", "s18", "s19"]

# ── RQ4 specific config ───────────────────────────────────────────────────────
DRIFT_CLIENT      = "FD001"   # which client experiences drift
DRIFT_ROUND       = 13        # which round drift is injected (midpoint of 25)
DRIFT_SOURCE      = "FD002"   # FD001 swaps to FD003 fault mode at drift round
DRIFT_THRESHOLD   = 0.03      # AUPRC drop that triggers drift detector
RECOVERY_LR       = 3e-3      # higher learning rate used during recovery
NORMAL_LR         = 1e-3      # normal learning rate

# ── paths ─────────────────────────────────────────────────────────────────────
BASE_DIR   = "F:/Mansu-FL/TASK-ERAU/C-MAPSS"
RQ4_RESULT_DIR = "F:/Mansu-FL/TASK-ERAU/results/FedAvg_RQ4"
os.makedirs(RQ4_RESULT_DIR, exist_ok=True)

FD_LIST = ["FD001", "FD002", "FD003", "FD004"]

print("✓ Config ready")
print(f"  FL rounds: {FL_ROUNDS} | Local epochs: {LOCAL_EPOCHS} | Window: {WINDOW_SIZE}")

print("✓ RQ4 config ready")
print(f"  Drift client : {DRIFT_CLIENT}")
print(f"  Drift round  : {DRIFT_ROUND}")
print(f"  Drift source : {DRIFT_SOURCE}")
print(f"  Threshold    : {DRIFT_THRESHOLD}")

✓ Config ready
  FL rounds: 25 | Local epochs: 8 | Window: 30
✓ RQ4 config ready
  Drift client : FD001
  Drift round  : 13
  Drift source : FD002
  Threshold    : 0.03


Data loading and preprocessing function

In [4]:
def load_client_data(fd_name):
    """Load, label, split, normalize and window one FD dataset."""

    DATA_DIR   = f"{BASE_DIR}/{fd_name}"
    train_path = f"{DATA_DIR}/train_{fd_name}.txt"
    test_path  = f"{DATA_DIR}/test_{fd_name}.txt"
    rul_path   = f"{DATA_DIR}/RUL_{fd_name}.txt"

    columns = (
        ["unit", "cycle"]
        + ["setting_1", "setting_2", "setting_3"]
        + [f"s{i}" for i in range(1, 22)]
    )

    # ── load ──────────────────────────────────────────────────────────────────
    train_df = pd.read_csv(train_path, sep=r"\s+", header=None, names=columns)
    test_df  = pd.read_csv(test_path,  sep=r"\s+", header=None, names=columns)
    rul_df   = pd.read_csv(rul_path,   sep=r"\s+", header=None, names=["RUL"])

    # ── train labels ──────────────────────────────────────────────────────────
    train_df["max_cycle"] = train_df.groupby("unit")["cycle"].transform("max")
    train_df["RUL_raw"]   = train_df["max_cycle"] - train_df["cycle"]
    train_df["RUL"]       = train_df["RUL_raw"].clip(upper=RUL_CAP)
    train_df["early_fault"] = (train_df["RUL_raw"] <= FAULT_THRESHOLD).astype(int)

    # ── test labels ───────────────────────────────────────────────────────────
    test_rul_map = dict(zip(range(1, len(rul_df) + 1), rul_df["RUL"].values))
    test_df["max_cycle"]  = test_df.groupby("unit")["cycle"].transform("max")
    test_df["final_RUL"]  = test_df["unit"].map(test_rul_map)
    test_df["RUL_raw"]    = test_df["final_RUL"] + (test_df["max_cycle"] - test_df["cycle"])
    test_df["RUL"]        = test_df["RUL_raw"].clip(upper=RUL_CAP)
    test_df["early_fault"] = (test_df["RUL_raw"] <= FAULT_THRESHOLD).astype(int)

    # ── train / val split ─────────────────────────────────────────────────────
    all_units = train_df["unit"].unique()
    train_units, val_units = train_test_split(all_units, test_size=0.2, random_state=SEED)
    train_part = train_df[train_df["unit"].isin(train_units)].copy()
    val_part   = train_df[train_df["unit"].isin(val_units)].copy()

    # ── feature columns ───────────────────────────────────────────────────────
    all_sensor_cols = [f"s{i}" for i in range(1, 22)]
    feature_cols = (
        ["setting_1", "setting_2", "setting_3"]
        + [s for s in all_sensor_cols if s not in DROP_SENSORS]
    )

    # ── normalization (auto-detects single vs multi-condition) ────────────────
    N_CONDITIONS = train_part[["setting_1", "setting_2", "setting_3"]].nunique().max()
    USE_CLUSTER_NORM = N_CONDITIONS > 2

    if USE_CLUSTER_NORM:
        km = KMeans(n_clusters=6, random_state=SEED, n_init=10)
        train_part["op_cluster"] = km.fit_predict(
            train_part[["setting_1", "setting_2", "setting_3"]]
        )
        scaler_dict = {}
        for c in range(6):
            mask = train_part["op_cluster"] == c
            sc   = StandardScaler()
            train_part.loc[mask, feature_cols] = sc.fit_transform(
                train_part.loc[mask, feature_cols]
            )
            scaler_dict[c] = sc

        def apply_cluster_scaler(df):
            df = df.copy()
            df["op_cluster"] = km.predict(df[["setting_1", "setting_2", "setting_3"]])
            for c, sc in scaler_dict.items():
                mask = df["op_cluster"] == c
                if mask.sum() > 0:
                    df.loc[mask, feature_cols] = sc.transform(df.loc[mask, feature_cols])
            return df

        val_part    = apply_cluster_scaler(val_part)
        test_scaled = apply_cluster_scaler(test_df.copy())
    else:
        sc = StandardScaler()
        train_part[feature_cols] = sc.fit_transform(train_part[feature_cols])
        val_part[feature_cols]   = sc.transform(val_part[feature_cols])
        test_scaled = test_df.copy()
        test_scaled[feature_cols] = sc.transform(test_scaled[feature_cols])

    # ── sliding windows ───────────────────────────────────────────────────────
    def make_windows(df):
        X, y_rul, y_fault, uids, ecycles = [], [], [], [], []
        for unit in sorted(df["unit"].unique()):
            udf = df[df["unit"] == unit].sort_values("cycle").reset_index(drop=True)
            if len(udf) < WINDOW_SIZE:
                continue
            feats  = udf[feature_cols].values
            ruls   = udf["RUL"].values
            faults = udf["early_fault"].values
            cycs   = udf["cycle"].values
            for s in range(len(udf) - WINDOW_SIZE + 1):
                e = s + WINDOW_SIZE
                X.append(feats[s:e])
                y_rul.append(ruls[e - 1])
                y_fault.append(faults[e - 1])
                uids.append(unit)
                ecycles.append(cycs[e - 1])
        return (
            np.array(X,       dtype=np.float32),
            np.array(y_rul,   dtype=np.float32),
            np.array(y_fault, dtype=np.float32),
            np.array(uids),
            np.array(ecycles)
        )

    X_tr, y_rul_tr, y_fault_tr, uids_tr, ec_tr = make_windows(train_part)
    X_vl, y_rul_vl, y_fault_vl, uids_vl, ec_vl = make_windows(val_part)
    X_te, y_rul_te, y_fault_te, uids_te, ec_te  = make_windows(test_scaled)

    norm_type = "cluster" if USE_CLUSTER_NORM else "standard"
    print(f"  {fd_name} | norm={norm_type} | "
          f"train={X_tr.shape} val={X_vl.shape} test={X_te.shape} | "
          f"fault ratio={y_fault_tr.mean():.3f}")

    return {
        "fd":           fd_name,
        "feature_cols": feature_cols,
        "X_train":      X_tr,  "y_rul_train":  y_rul_tr,  "y_fault_train":  y_fault_tr,
        "X_val":        X_vl,  "y_rul_val":    y_rul_vl,  "y_fault_val":    y_fault_vl,
        "X_test":       X_te,  "y_rul_test":   y_rul_te,  "y_fault_test":   y_fault_te,
        "unit_ids_test": uids_te, "end_cycles_test": ec_te
    }


# ── load all 4 clients ────────────────────────────────────────────────────────
print("Loading all clients...")
clients = {fd: load_client_data(fd) for fd in FD_LIST}
print("✓ All clients loaded")

Loading all clients...


C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-1.44839748 -1.44839748 -2.09663281 ...  0.49630848  1.79277913
  2.44101445]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_part.loc[mask, feature_cols] = sc.fit_transform(
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.80016216 -0.80016216 -1.44839748 -0.15192684 -0.80016216 -0.80016216
 -0.80016216 -1.44839748 -0.80016216  0.49630848 -1.44839748 -0.15192684
 -1.44839748 -0.80016216 -0.15192684  0.49630848 -0.80016216 -0.80016216
 -1.44839748  0.49630848  0.49630848 -0.15192684 -0.15192684 -0.15192684
 -0.80016216 -0.15192684 -0.80016216 -0.15192684 -0.80016216 -0.15192684
  0.49630848  1.144

  FD001 | norm=cluster | train=(14241, 30, 18) val=(3490, 30, 18) test=(10196, 30, 18) | fault ratio=0.174


C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.76383951 -1.47678703 -0.76383951 ...  2.08795057  2.08795057
  2.08795057]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_part.loc[mask, feature_cols] = sc.fit_transform(
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-2.18973455 -0.76383951 -1.47678703 ...  2.08795057  2.08795057
  1.37500305]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[mask, feature_cols] = sc.transform(df.loc[mask, feature_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will ra

  FD002 | norm=cluster | train=(37432, 30, 18) val=(8787, 30, 18) test=(26505, 30, 18) | fault ratio=0.172


C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.31207787 -0.31207787 -0.31207787 ...  0.82466606  0.82466606
  2.52978195]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_part.loc[mask, feature_cols] = sc.fit_transform(
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.88044983 -0.88044983 -0.88044983 ...  1.96140999  1.96140999
  1.96140999]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[mask, feature_cols] = sc.transform(df.loc[mask, feature_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will ra

  FD003 | norm=cluster | train=(17692, 30, 18) val=(4128, 30, 18) test=(13696, 30, 18) | fault ratio=0.140


C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-2.27190928 -1.04409768 -1.65800348 ...  1.41152551  1.41152551
  2.0254313 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_part.loc[mask, feature_cols] = sc.fit_transform(
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.79761971 -0.43019189  0.18371391 ... -0.43019189  0.79761971
  0.79761971]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[mask, feature_cols] = sc.transform(df.loc[mask, feature_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will ra

  FD004 | norm=cluster | train=(43523, 30, 18) val=(10505, 30, 18) test=(34081, 30, 18) | fault ratio=0.142
✓ All clients loaded


In [5]:
def load_client_data_with_drift(fd_name, drift_fd=None):
    """
    Same as load_client_data but optionally replaces training data
    with drift_fd data to simulate a fault mode shift.

    fd_name  : original client dataset (e.g. FD001)
    drift_fd : if not None, replace train/val with this dataset (e.g. FD003)
               but keep test set from original fd_name for fair evaluation
    """
    # load original data for test set
    original = load_client_data(fd_name)

    if drift_fd is None:
        return original

    # load drift source data for train/val
    drifted = load_client_data(drift_fd)

    # replace train and val with drifted data
    # keep test from original so evaluation is consistent
    original["X_train"]       = drifted["X_train"]
    original["y_rul_train"]   = drifted["y_rul_train"]
    original["y_fault_train"] = drifted["y_fault_train"]
    original["X_val"]         = drifted["X_val"]
    original["y_rul_val"]     = drifted["y_rul_val"]
    original["y_fault_val"]   = drifted["y_fault_val"]

    print(f"  ⚠ Drift applied: {fd_name} training data replaced with {drift_fd}")
    print(f"    New fault ratio: {drifted['y_fault_train'].mean():.3f} "
          f"(was {original['y_fault_train'].mean():.3f})")

    return original


# ── pre-load both normal and drifted versions of DRIFT_CLIENT ─────────────────
print("Loading all clients (normal)...")
clients_normal = {fd: load_client_data(fd) for fd in FD_LIST}

print("\nLoading drifted version of drift client...")
clients_drifted = dict(clients_normal)   # copy
clients_drifted[DRIFT_CLIENT] = load_client_data_with_drift(
    DRIFT_CLIENT, drift_fd=DRIFT_SOURCE
)

print("\n✓ Normal and drifted client data ready")
print(f"  Normal  {DRIFT_CLIENT} fault ratio: "
      f"{clients_normal[DRIFT_CLIENT]['y_fault_train'].mean():.3f}")
print(f"  Drifted {DRIFT_CLIENT} fault ratio: "
      f"{clients_drifted[DRIFT_CLIENT]['y_fault_train'].mean():.3f}")

Loading all clients (normal)...


C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-1.44839748 -1.44839748 -2.09663281 ...  0.49630848  1.79277913
  2.44101445]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_part.loc[mask, feature_cols] = sc.fit_transform(
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.80016216 -0.80016216 -1.44839748 -0.15192684 -0.80016216 -0.80016216
 -0.80016216 -1.44839748 -0.80016216  0.49630848 -1.44839748 -0.15192684
 -1.44839748 -0.80016216 -0.15192684  0.49630848 -0.80016216 -0.80016216
 -1.44839748  0.49630848  0.49630848 -0.15192684 -0.15192684 -0.15192684
 -0.80016216 -0.15192684 -0.80016216 -0.15192684 -0.80016216 -0.15192684
  0.49630848  1.144

  FD001 | norm=cluster | train=(14241, 30, 18) val=(3490, 30, 18) test=(10196, 30, 18) | fault ratio=0.174


C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.76383951 -1.47678703 -0.76383951 ...  2.08795057  2.08795057
  2.08795057]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_part.loc[mask, feature_cols] = sc.fit_transform(
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-2.18973455 -0.76383951 -1.47678703 ...  2.08795057  2.08795057
  1.37500305]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[mask, feature_cols] = sc.transform(df.loc[mask, feature_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will ra

  FD002 | norm=cluster | train=(37432, 30, 18) val=(8787, 30, 18) test=(26505, 30, 18) | fault ratio=0.172


...

  FD004 | norm=cluster | train=(43523, 30, 18) val=(10505, 30, 18) test=(34081, 30, 18) | fault ratio=0.142

Loading drifted version of drift client...


C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-1.44839748 -1.44839748 -2.09663281 ...  0.49630848  1.79277913
  2.44101445]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_part.loc[mask, feature_cols] = sc.fit_transform(
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.80016216 -0.80016216 -1.44839748 -0.15192684 -0.80016216 -0.80016216
 -0.80016216 -1.44839748 -0.80016216  0.49630848 -1.44839748 -0.15192684
 -1.44839748 -0.80016216 -0.15192684  0.49630848 -0.80016216 -0.80016216
 -1.44839748  0.49630848  0.49630848 -0.15192684 -0.15192684 -0.15192684
 -0.80016216 -0.15192684 -0.80016216 -0.15192684 -0.80016216 -0.15192684
  0.49630848  1.144

  FD001 | norm=cluster | train=(14241, 30, 18) val=(3490, 30, 18) test=(10196, 30, 18) | fault ratio=0.174


C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.76383951 -1.47678703 -0.76383951 ...  2.08795057  2.08795057
  2.08795057]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_part.loc[mask, feature_cols] = sc.fit_transform(
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-2.18973455 -0.76383951 -1.47678703 ...  2.08795057  2.08795057
  1.37500305]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[mask, feature_cols] = sc.transform(df.loc[mask, feature_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3253605258.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will ra

  FD002 | norm=cluster | train=(37432, 30, 18) val=(8787, 30, 18) test=(26505, 30, 18) | fault ratio=0.172
  ⚠ Drift applied: FD001 training data replaced with FD002
    New fault ratio: 0.172 (was 0.172)

✓ Normal and drifted client data ready
  Normal  FD001 fault ratio: 0.174
  Drifted FD001 fault ratio: 0.172


Model builder and loss function

In [6]:
def weighted_bce(pos_weight):
    """Weighted binary crossentropy to handle class imbalance."""
    def loss_fn(y_true, y_pred):
        bce    = tf.keras.backend.binary_crossentropy(y_true, y_pred)
        weight = y_true * pos_weight + (1 - y_true) * 1.0
        return tf.reduce_mean(weight * bce)
    return loss_fn


def build_model(input_shape, pos_weight):
    """
    1D-CNN → BiLSTM → MultiHeadAttention → Dual output heads
    Same architecture as your centralized model.
    """
    inputs = layers.Input(shape=input_shape)

    # CNN block
    x = layers.Conv1D(64,  kernel_size=5, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)

    # BiLSTM + attention
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
    x = layers.GlobalAveragePooling1D()(x)

    # Dense head
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.2)(x)

    # Dual outputs
    rul_output   = layers.Dense(1, activation="linear",  name="rul_output")(x)
    fault_output = layers.Dense(1, activation="sigmoid", name="fault_output")(x)

    model = models.Model(inputs=inputs, outputs=[rul_output, fault_output])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss={
            "rul_output":   "mse",
            "fault_output": weighted_bce(pos_weight)
        },
        loss_weights={
            "rul_output":   1.0,
            "fault_output": 1.5
        },
        metrics={
            "rul_output":   ["mae"],
            "fault_output": [
                "accuracy",
                tf.keras.metrics.AUC(curve="PR", name="auprc")
            ]
        }
    )
    return model


def compute_pos_weight(y_fault):
    """sqrt-dampened pos_weight to avoid collapse."""
    neg = np.sum(y_fault == 0)
    pos = np.sum(y_fault == 1)
    return float(np.sqrt(neg / pos)) if pos > 0 else 1.0


# ── verify: build one model per client and print summary ─────────────────────
print("Building one model per client...\n")
input_shape = (WINDOW_SIZE, len(clients["FD001"]["feature_cols"]))

for fd, client in clients.items():
    pw = compute_pos_weight(client["y_fault_train"])
    m  = build_model(input_shape, pw)
    print(f"  {fd} | pos_weight={pw:.3f} | params={m.count_params():,}")

print("\n✓ Model builder ready")

Building one model per client...

  FD001 | pos_weight=2.178 | params=221,378
  FD002 | pos_weight=2.192 | params=221,378
  FD003 | pos_weight=2.477 | params=221,378
  FD004 | pos_weight=2.461 | params=221,378

✓ Model builder ready


FedAvg server (weight averaging logic)

In [7]:
def fedavg_aggregate(global_model, client_models, client_sizes, client_fault_ratios=None, alpha=0.5):
    """
    Fault-aware FedAvg aggregation (RQ2).

    client_sizes        : list of ints — training windows per client
    client_fault_ratios : list of floats — fault ratio per client
                          if None → falls back to standard FedAvg
    alpha               : 0.0 = pure fault-aware
                          1.0 = pure size-based (standard FedAvg)
                          0.5 = balanced (default)
    """
    n = len(client_models)

    # ── size-based weights (standard FedAvg) ─────────────────────────────────
    total         = sum(client_sizes)
    size_weights  = np.array([s / total for s in client_sizes])

    # ── fault-aware weights ───────────────────────────────────────────────────
    if client_fault_ratios is not None:
        eps          = 1e-6
        inv_fault    = np.array([1.0 / (r + eps) for r in client_fault_ratios])
        fault_weights = inv_fault / inv_fault.sum()   # normalize to sum=1
    else:
        fault_weights = size_weights                   # fallback to size

    # ── combined weights ──────────────────────────────────────────────────────
    final_weights = alpha * size_weights + (1 - alpha) * fault_weights
    final_weights = final_weights / final_weights.sum()   # re-normalize

    # ── log weights for transparency ─────────────────────────────────────────
    for i, fd in enumerate(FD_LIST):
        print(f"    {fd} → size_w={size_weights[i]:.3f}  "
              f"fault_w={fault_weights[i]:.3f}  "
              f"final_w={final_weights[i]:.3f}")

    # ── weighted average layer by layer ──────────────────────────────────────
    all_weights = [m.get_weights() for m in client_models]
    avg_weights = []
    for layer_idx in range(len(all_weights[0])):
        layer_avg = np.sum(
            [final_weights[i] * all_weights[i][layer_idx] for i in range(n)],
            axis=0
        )
        avg_weights.append(layer_avg)

    global_model.set_weights(avg_weights)
    return global_model


def get_client_model(global_model, input_shape, pos_weight):
    """Copy global weights into a fresh client model."""
    client_model = build_model(input_shape, pos_weight)
    client_model.set_weights(global_model.get_weights())
    return client_model


# ── sanity check ──────────────────────────────────────────────────────────────
print("Sanity check — fault-aware FedAvg aggregation...\n")

input_shape  = (WINDOW_SIZE, len(clients["FD001"]["feature_cols"]))
pw_test      = compute_pos_weight(clients["FD001"]["y_fault_train"])
test_global  = build_model(input_shape, pw_test)
cm1 = build_model(input_shape, pw_test)
cm2 = build_model(input_shape, pw_test)
cm3 = build_model(input_shape, pw_test)
cm4 = build_model(input_shape, pw_test)

# simulate 4 clients with different fault ratios
test_sizes  = [8000, 20000, 8000, 18000]
test_ratios = [0.18, 0.15, 0.12, 0.10]   # FD003/FD004 rarer faults → get boosted

print("  Example aggregation weights:")
test_global = fedavg_aggregate(
    test_global,
    [cm1, cm2, cm3, cm4],
    test_sizes,
    client_fault_ratios=test_ratios,
    alpha=0.5
)

print("\n✓ Fault-aware aggregation ready")
del test_global, cm1, cm2, cm3, cm4

Sanity check — fault-aware FedAvg aggregation...

  Example aggregation weights:
    FD001 → size_w=0.148  fault_w=0.182  final_w=0.165
    FD002 → size_w=0.370  fault_w=0.218  final_w=0.294
    FD003 → size_w=0.148  fault_w=0.273  final_w=0.210
    FD004 → size_w=0.333  fault_w=0.327  final_w=0.330

✓ Fault-aware aggregation ready


FL training loop

In [8]:
def train_one_round(global_model, clients, input_shape, round_num):
    """
    One FL round with fault-aware aggregation (RQ2).
    Passes per-client fault ratios into fedavg_aggregate.
    """
    client_models       = []
    client_sizes        = []
    client_fault_ratios = []
    round_logs          = {}

    for fd, client in clients.items():
        # ── copy global weights to client ─────────────────────────────────────
        pw           = compute_pos_weight(client["y_fault_train"])
        client_model = get_client_model(global_model, input_shape, pw)

        # ── local training ────────────────────────────────────────────────────
        client_model.fit(
            client["X_train"],
            {
                "rul_output":   client["y_rul_train"]  / RUL_CAP,
                "fault_output": client["y_fault_train"]
            },
            validation_data=(
                client["X_val"],
                {
                    "rul_output":   client["y_rul_val"] / RUL_CAP,
                    "fault_output": client["y_fault_val"]
                }
            ),
            epochs     = LOCAL_EPOCHS,
            batch_size = BATCH_SIZE,
            verbose    = 0
        )

        # ── collect ───────────────────────────────────────────────────────────
        client_models.append(client_model)
        client_sizes.append(len(client["X_train"]))

        # fault ratio for this client
        fault_ratio = float(client["y_fault_train"].mean())
        client_fault_ratios.append(fault_ratio)

        # val loss for logging
        val_loss = client_model.evaluate(
            client["X_val"],
            {
                "rul_output":   client["y_rul_val"] / RUL_CAP,
                "fault_output": client["y_fault_val"]
            },
            verbose=0
        )[0]
        round_logs[fd] = {
            "val_loss":   val_loss,
            "fault_ratio": fault_ratio
        }

    # ── fault-aware aggregation ───────────────────────────────────────────────
    print(f"\n  Round {round_num:02d} — aggregation weights:")
    global_model = fedavg_aggregate(
        global_model,
        client_models,
        client_sizes,
        client_fault_ratios=client_fault_ratios,
        alpha=0.5
    )

    # round summary
    log_str = "  " + " | ".join(
        [f"{fd} loss={v['val_loss']:.4f} fault_ratio={v['fault_ratio']:.3f}"
         for fd, v in round_logs.items()]
    )
    print(log_str)

    return global_model, round_logs


print("✓ RQ2 training loop ready")

✓ RQ2 training loop ready


In [9]:
class DriftDetector:
    """
    Monitors per-client AUPRC each round.
    Triggers if AUPRC drops more than DRIFT_THRESHOLD
    compared to the best seen so far.
    """
    def __init__(self, threshold=DRIFT_THRESHOLD):
        self.threshold    = threshold
        self.best_auprc   = {}    # best AUPRC seen per client
        self.drift_detected = {}  # round drift was detected per client
        self.recovering   = {}    # is client currently in recovery mode

    def update(self, fd, auprc, round_num):
        """
        Call every round with current client AUPRC.
        Returns True if drift detected or currently recovering.
        """
        # initialize on first call
        if fd not in self.best_auprc:
            self.best_auprc[fd]     = auprc
            self.drift_detected[fd] = None
            self.recovering[fd]     = False
            return False

        # update best
        if auprc > self.best_auprc[fd]:
            self.best_auprc[fd] = auprc
            self.recovering[fd] = False   # recovered

        # check for drop
        drop = self.best_auprc[fd] - auprc
        if drop >= self.threshold and self.drift_detected[fd] is None:
            self.drift_detected[fd] = round_num
            self.recovering[fd]     = True
            print(f"    🚨 DRIFT DETECTED on {fd} at round {round_num} "
                  f"(AUPRC dropped {drop:.4f} from best {self.best_auprc[fd]:.4f})")

        return self.recovering[fd]

    def status(self, fd):
        return {
            "best_auprc":     self.best_auprc.get(fd, None),
            "drift_round":    self.drift_detected.get(fd, None),
            "recovering":     self.recovering.get(fd, False)
        }


# ── quick test ────────────────────────────────────────────────────────────────
print("Testing drift detector...")
detector = DriftDetector(threshold=0.05)
test_auprcs = [0.89, 0.90, 0.91, 0.91, 0.90, 0.84, 0.83, 0.85, 0.88, 0.91]
for rnd, val in enumerate(test_auprcs, 1):
    triggered = detector.update("FD001", val, rnd)
    print(f"  Round {rnd:02d} AUPRC={val:.2f} | "
          f"best={detector.best_auprc['FD001']:.2f} | "
          f"recovering={triggered}")

print("\n✓ Drift detector ready")

Testing drift detector...
  Round 01 AUPRC=0.89 | best=0.89 | recovering=False
  Round 02 AUPRC=0.90 | best=0.90 | recovering=False
  Round 03 AUPRC=0.91 | best=0.91 | recovering=False
  Round 04 AUPRC=0.91 | best=0.91 | recovering=False
  Round 05 AUPRC=0.90 | best=0.91 | recovering=False
    🚨 DRIFT DETECTED on FD001 at round 6 (AUPRC dropped 0.0700 from best 0.9100)
  Round 06 AUPRC=0.84 | best=0.91 | recovering=True
  Round 07 AUPRC=0.83 | best=0.91 | recovering=True
  Round 08 AUPRC=0.85 | best=0.91 | recovering=True
  Round 09 AUPRC=0.88 | best=0.91 | recovering=True
  Round 10 AUPRC=0.91 | best=0.91 | recovering=True

✓ Drift detector ready


In [10]:
def train_one_round_rq4(global_model, clients, input_shape, round_num,
                         detector, auprc_history):
    """
    One FL round with:
      - Drift injection at DRIFT_ROUND (swap DRIFT_CLIENT data)
      - Per-client AUPRC monitoring every round
      - Recovery LR boost when drift detected
    """
    client_models       = []
    client_sizes        = []
    client_fault_ratios = []
    round_logs          = {}

    # ── decide which client data to use this round ────────────────────────────
    if round_num >= DRIFT_ROUND:
        active_clients = clients_drifted   # drifted version
        if round_num == DRIFT_ROUND:
            print(f"\n  ⚠ Round {round_num}: DRIFT INJECTED on {DRIFT_CLIENT} "
                  f"(data swapped to {DRIFT_SOURCE})")
    else:
        active_clients = clients_normal    # normal version

    for fd, client in active_clients.items():
        # ── check drift status for this client ────────────────────────────────
        is_recovering = detector.recovering.get(fd, False)

        # ── set learning rate based on drift status ───────────────────────────
        lr = RECOVERY_LR if is_recovering else NORMAL_LR

        # ── copy global weights to client ─────────────────────────────────────
        pw           = compute_pos_weight(client["y_fault_train"])
        client_model = get_client_model(global_model, input_shape, pw)

        # update LR if recovering
        client_model.optimizer.learning_rate.assign(lr)


        # ── local training ────────────────────────────────────────────────────
        client_model.fit(
            client["X_train"],
            {
                "rul_output":   client["y_rul_train"]  / RUL_CAP,
                "fault_output": client["y_fault_train"]
            },
            validation_data=(
                client["X_val"],
                {
                    "rul_output":   client["y_rul_val"] / RUL_CAP,
                    "fault_output": client["y_fault_val"]
                }
            ),
            epochs     = LOCAL_EPOCHS,
            batch_size = BATCH_SIZE,
            verbose    = 0
        )

        # ── evaluate val AUPRC for drift monitoring ───────────────────────────
        pred_val = client_model.predict(client["X_val"], verbose=0)
        pred_fault_prob_val = pred_val[1].flatten()
        try:
            val_auprc = average_precision_score(
                client["y_fault_val"], pred_fault_prob_val
            )
        except:
            val_auprc = 0.0

        # ── update drift detector ─────────────────────────────────────────────
        detector.update(fd, val_auprc, round_num)
        auprc_history[fd].append(val_auprc)

        # ── collect ───────────────────────────────────────────────────────────
        client_models.append(client_model)
        client_sizes.append(len(client["X_train"]))
        client_fault_ratios.append(float(client["y_fault_train"].mean()))

        val_loss = client_model.evaluate(
            client["X_val"],
            {
                "rul_output":   client["y_rul_val"] / RUL_CAP,
                "fault_output": client["y_fault_val"]
            },
            verbose=0
        )[0]

        round_logs[fd] = {
            "val_loss":    val_loss,
            "val_auprc":   val_auprc,
            "recovering":  is_recovering,
            "lr":          lr
        }

    # ── standard FedAvg aggregation (no fault-aware for RQ4 baseline) ─────────
    global_model = fedavg_aggregate(
        global_model,
        client_models,
        client_sizes,
        client_fault_ratios=None   # plain FedAvg for RQ4
    )

    # ── round summary ─────────────────────────────────────────────────────────
    print(f"  Round {round_num:02d} →", end="")
    for fd, log in round_logs.items():
        marker = "🔴" if fd == DRIFT_CLIENT and round_num >= DRIFT_ROUND else "  "
        recov  = " [RECOVERING]" if log["recovering"] else ""
        print(f"  {marker}{fd} loss={log['val_loss']:.4f} "
              f"auprc={log['val_auprc']:.4f}{recov}", end="")
    print()

    return global_model, round_logs


print("✓ RQ4 training loop ready")

✓ RQ4 training loop ready


In [11]:
def evaluate_client(global_model, client, input_shape):
    pw    = compute_pos_weight(client["y_fault_train"])
    model = get_client_model(global_model, input_shape, pw)

    pred_rul_scaled, pred_fault_prob = model.predict(client["X_test"], verbose=0)
    pred_rul        = pred_rul_scaled.flatten() * RUL_CAP
    pred_fault_prob = pred_fault_prob.flatten()
    pred_fault      = (pred_fault_prob >= 0.5).astype(int)

    y_rul   = client["y_rul_test"]
    y_fault = client["y_fault_test"]

    mae  = mean_absolute_error(y_rul, pred_rul)
    rmse = np.sqrt(mean_squared_error(y_rul, pred_rul))
    r2   = r2_score(y_rul, pred_rul)
    acc  = accuracy_score(y_fault, pred_fault)
    prec = precision_score(y_fault, pred_fault, zero_division=0)
    rec  = recall_score(y_fault, pred_fault, zero_division=0)
    f1   = f1_score(y_fault, pred_fault, zero_division=0)
    auprc = average_precision_score(y_fault, pred_fault_prob)
    try:    auroc = roc_auc_score(y_fault, pred_fault_prob)
    except: auroc = 0.0

    info = pd.DataFrame({
        "idx":       np.arange(len(client["unit_ids_test"])),
        "unit":      client["unit_ids_test"],
        "end_cycle": client["end_cycles_test"]
    })
    last_idx    = info.sort_values("end_cycle").groupby("unit")["idx"].last().values
    X_last      = client["X_test"][last_idx]
    y_rul_last  = client["y_rul_test"][last_idx]
    y_fault_last = client["y_fault_test"][last_idx]

    pr_scaled_last, pf_prob_last = model.predict(X_last, verbose=0)
    pr_last  = pr_scaled_last.flatten() * RUL_CAP
    pf_last  = (pf_prob_last.flatten() >= 0.5).astype(int)
    pfp_last = pf_prob_last.flatten()

    mae_l  = mean_absolute_error(y_rul_last, pr_last)
    rmse_l = np.sqrt(mean_squared_error(y_rul_last, pr_last))
    r2_l   = r2_score(y_rul_last, pr_last)
    acc_l  = accuracy_score(y_fault_last, pf_last)
    prec_l = precision_score(y_fault_last, pf_last, zero_division=0)
    rec_l  = recall_score(y_fault_last, pf_last, zero_division=0)
    f1_l   = f1_score(y_fault_last, pf_last, zero_division=0)
    auprc_l = average_precision_score(y_fault_last, pfp_last)
    try:    auroc_l = roc_auc_score(y_fault_last, pfp_last)
    except: auroc_l = 0.0

    return {
        "all":  dict(mae=mae,  rmse=rmse,  r2=r2,  acc=acc,  prec=prec,
                     rec=rec,  f1=f1,  auprc=auprc,  auroc=auroc),
        "last": dict(mae=mae_l, rmse=rmse_l, r2=r2_l, acc=acc_l, prec=prec_l,
                     rec=rec_l, f1=f1_l, auprc=auprc_l, auroc=auroc_l)
    }

print("✓ evaluate_client ready")

✓ evaluate_client ready


Run the full FL experiment and save results

In [12]:
# ── setup ─────────────────────────────────────────────────────────────────────
input_shape  = (WINDOW_SIZE, len(clients_normal["FD001"]["feature_cols"]))
global_pw    = compute_pos_weight(clients_normal["FD001"]["y_fault_train"])
global_model = build_model(input_shape, global_pw)

# storage
auprc_history = {fd: [] for fd in FD_LIST}   # val AUPRC per round per client
round_history = {fd: [] for fd in FD_LIST}   # val loss per round
detector      = DriftDetector(threshold=DRIFT_THRESHOLD)
final_metrics = {}

print("=" * 60)
print(f"Starting FedAvg RQ4 | {FL_ROUNDS} rounds | {LOCAL_EPOCHS} local epochs")
print(f"Drift injection: {DRIFT_CLIENT} → {DRIFT_SOURCE} at round {DRIFT_ROUND}")
print("=" * 60)

# ── FL rounds ─────────────────────────────────────────────────────────────────
for rnd in range(1, FL_ROUNDS + 1):
    global_model, round_logs = train_one_round_rq4(
        global_model, clients_normal, input_shape, rnd,
        detector, auprc_history
    )
    for fd in FD_LIST:
        round_history[fd].append(round_logs[fd]["val_loss"])

print("\n✓ Training complete — evaluating all clients...")

# ── final evaluation ──────────────────────────────────────────────────────────
for fd in FD_LIST:
    final_metrics[fd] = evaluate_client(
        global_model, clients_normal[fd], input_shape
    )
    m = final_metrics[fd]
    print(f"\n  {fd}")
    print(f"    All windows → MAE={m['all']['mae']:.2f}  R²={m['all']['r2']:.3f}  "
          f"Rec={m['all']['rec']:.3f}  F1={m['all']['f1']:.3f}  "
          f"AUPRC={m['all']['auprc']:.3f}")
    print(f"    Last window → MAE={m['last']['mae']:.2f}  R²={m['last']['r2']:.3f}  "
          f"Rec={m['last']['rec']:.3f}  F1={m['last']['f1']:.3f}  "
          f"AUPRC={m['last']['auprc']:.3f}")

# ── drift detector summary ────────────────────────────────────────────────────
print("\n── Drift Detector Summary ───────────────────────────────")
for fd in FD_LIST:
    s = detector.status(fd)
    print(f"  {fd} | best_auprc={s['best_auprc']:.4f} | "
          f"drift_round={s['drift_round']} | "
          f"recovered={not s['recovering']}")

# ── plot 1: AUPRC per round — main RQ4 evidence ───────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
colors = {"FD001": "red", "FD002": "steelblue", "FD003": "green", "FD004": "orange"}

for fd in FD_LIST:
    lw = 2.5 if fd == DRIFT_CLIENT else 1.2
    ax.plot(range(1, FL_ROUNDS + 1), auprc_history[fd],
            label=fd, color=colors[fd], linewidth=lw,
            marker="o", markersize=3)

# mark drift injection point
ax.axvline(x=DRIFT_ROUND, color="black", linestyle="--", linewidth=1.5,
           label=f"Drift injected (round {DRIFT_ROUND})")

# mark drift detection point
drift_rnd = detector.status(DRIFT_CLIENT)["drift_round"]
if drift_rnd:
    ax.axvline(x=drift_rnd, color="red", linestyle=":", linewidth=1.5,
               label=f"Drift detected (round {drift_rnd})")

    # shade recovery window
    recovery_end = min(drift_rnd + 10, FL_ROUNDS)
    ax.axvspan(drift_rnd, recovery_end, alpha=0.1, color="red",
               label=f"10-round recovery window")

ax.set_xlabel("FL Round")
ax.set_ylabel("Validation AUPRC")
ax.set_title(f"RQ4 — AUPRC across FL rounds\n"
             f"Drift injected on {DRIFT_CLIENT} at round {DRIFT_ROUND} "
             f"(data shifted to {DRIFT_SOURCE})")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(RQ4_RESULT_DIR, "rq4_auprc_per_round.png"),
            dpi=150, bbox_inches="tight")
plt.show(); plt.close()

# ── plot 2: val loss per round ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
for fd in FD_LIST:
    lw = 2.5 if fd == DRIFT_CLIENT else 1.2
    ax.plot(range(1, FL_ROUNDS + 1), round_history[fd],
            label=fd, color=colors[fd], linewidth=lw,
            marker="o", markersize=3)
ax.axvline(x=DRIFT_ROUND, color="black", linestyle="--", linewidth=1.5,
           label=f"Drift injected (round {DRIFT_ROUND})")
ax.set_xlabel("FL Round")
ax.set_ylabel("Validation Loss")
ax.set_title("RQ4 — validation loss across FL rounds")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RQ4_RESULT_DIR, "rq4_val_loss_per_round.png"),
            dpi=150, bbox_inches="tight")
plt.show(); plt.close()

# ── plot 3: zoom on drift client AUPRC ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, FL_ROUNDS + 1), auprc_history[DRIFT_CLIENT],
        color="red", linewidth=2.5, marker="o", markersize=4,
        label=f"{DRIFT_CLIENT} (drift client)")
ax.axvline(x=DRIFT_ROUND, color="black", linestyle="--",
           label=f"Drift injected (round {DRIFT_ROUND})")
if drift_rnd:
    ax.axvline(x=drift_rnd, color="red", linestyle=":",
               label=f"Drift detected (round {drift_rnd})")
    recovery_end = min(drift_rnd + 10, FL_ROUNDS)
    ax.axvspan(drift_rnd, recovery_end, alpha=0.15, color="red",
               label="10-round recovery window")
    # annotate recovery
    recovered_auprc = auprc_history[DRIFT_CLIENT][recovery_end - 1]
    ax.annotate(f"AUPRC={recovered_auprc:.3f}\nat round {recovery_end}",
                xy=(recovery_end, recovered_auprc),
                xytext=(recovery_end - 3, recovered_auprc - 0.08),
                arrowprops=dict(arrowstyle="->", color="black"),
                fontsize=9)
ax.set_xlabel("FL Round")
ax.set_ylabel("Validation AUPRC")
ax.set_title(f"RQ4 — {DRIFT_CLIENT} AUPRC degradation and recovery")
ax.legend(); ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(RQ4_RESULT_DIR, "rq4_drift_client_zoom.png"),
            dpi=150, bbox_inches="tight")
plt.show(); plt.close()

# ── save metrics txt ──────────────────────────────────────────────────────────
lines = [
    f"FedAvg RQ4 — Concept Drift Detection and Recovery\n",
    f"{FL_ROUNDS} rounds | {LOCAL_EPOCHS} local epochs\n",
    f"Drift client: {DRIFT_CLIENT} | Drift round: {DRIFT_ROUND} | "
    f"Source: {DRIFT_SOURCE}\n",
    f"Drift threshold: {DRIFT_THRESHOLD} | Recovery LR: {RECOVERY_LR}\n",
    "=" * 55 + "\n",
    "\nDrift Detector Summary:\n"
]

for fd in FD_LIST:
    s = detector.status(fd)
    lines.append(f"  {fd}: best_auprc={s['best_auprc']:.4f}  "
                 f"drift_detected_round={s['drift_round']}  "
                 f"recovered={not s['recovering']}\n")

if drift_rnd:
    pre_drift_auprc  = auprc_history[DRIFT_CLIENT][DRIFT_ROUND - 2]
    post_drift_auprc = min(auprc_history[DRIFT_CLIENT][DRIFT_ROUND:])
    recovery_auprc   = auprc_history[DRIFT_CLIENT][min(drift_rnd + 9, FL_ROUNDS) - 1]
    lines += [
        f"\nDrift Client ({DRIFT_CLIENT}) AUPRC timeline:\n",
        f"  Pre-drift  (round {DRIFT_ROUND-1}): {pre_drift_auprc:.4f}\n",
        f"  Min post-drift:                    {post_drift_auprc:.4f}\n",
        f"  After 10 recovery rounds:          {recovery_auprc:.4f}\n",
        f"  Drop:                              "
        f"{pre_drift_auprc - post_drift_auprc:.4f}\n",
        f"  Recovery gain:                     "
        f"{recovery_auprc - post_drift_auprc:.4f}\n",
    ]

lines.append("\n" + "=" * 55 + "\n")
for fd in FD_LIST:
    m = final_metrics[fd]
    lines.append(f"\n{fd}\n")
    lines.append("  -- All windows --\n")
    for k, v in m["all"].items():
        lines.append(f"  {k:<6}: {v:.4f}\n")
    lines.append("  -- Last window --\n")
    for k, v in m["last"].items():
        lines.append(f"  {k:<6}: {v:.4f}\n")

with open(os.path.join(RQ4_RESULT_DIR, "rq4_metrics.txt"), "w") as f:
    f.writelines(lines)

# ── save model ────────────────────────────────────────────────────────────────
global_model.save(os.path.join(RQ4_RESULT_DIR, "fedavg_rq4_global_model.keras"))

print(f"\n✓ All RQ4 results saved to: {RQ4_RESULT_DIR}")
print("  rq4_auprc_per_round.png      ← main RQ4 evidence")
print("  rq4_drift_client_zoom.png    ← zoomed drift + recovery")
print("  rq4_val_loss_per_round.png")
print("  rq4_metrics.txt")
print("  fedavg_rq4_global_model.keras")

Starting FedAvg RQ4 | 25 rounds | 8 local epochs
Drift injection: FD001 → FD002 at round 13
    FD001 → size_w=0.126  fault_w=0.126  final_w=0.126
    FD002 → size_w=0.332  fault_w=0.332  final_w=0.332
    FD003 → size_w=0.157  fault_w=0.157  final_w=0.157
    FD004 → size_w=0.386  fault_w=0.386  final_w=0.386
  Round 01 →    FD001 loss=0.3959 auprc=0.9658    FD002 loss=0.2711 auprc=0.9620    FD003 loss=0.3016 auprc=0.9321    FD004 loss=0.3074 auprc=0.9283
    FD001 → size_w=0.126  fault_w=0.126  final_w=0.126
    FD002 → size_w=0.332  fault_w=0.332  final_w=0.332
    FD003 → size_w=0.157  fault_w=0.157  final_w=0.157
    FD004 → size_w=0.386  fault_w=0.386  final_w=0.386
  Round 02 →    FD001 loss=0.3285 auprc=0.9642    FD002 loss=0.3610 auprc=0.9493    FD003 loss=0.2780 auprc=0.9501    FD004 loss=0.3140 auprc=0.9330
    FD001 → size_w=0.126  fault_w=0.126  final_w=0.126
    FD002 → size_w=0.332  fault_w=0.332  final_w=0.332
    FD003 → size_w=0.157  fault_w=0.157  final_w=0.157
    F

C:\Users\User\AppData\Local\Temp\ipykernel_25512\3611155462.py:86: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()
C:\Users\User\AppData\Local\Temp\ipykernel_25512\3611155462.py:104: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()



✓ All RQ4 results saved to: F:/Mansu-FL/TASK-ERAU/results/FedAvg_RQ4
  rq4_auprc_per_round.png      ← main RQ4 evidence
  rq4_drift_client_zoom.png    ← zoomed drift + recovery
  rq4_val_loss_per_round.png
  rq4_metrics.txt
  fedavg_rq4_global_model.keras


C:\Users\User\AppData\Local\Temp\ipykernel_25512\3611155462.py:134: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()
